In [1]:
import cda2
import json
import re
import pyspark.sql.types as T
import pyspark.sql.functions as F
#import datetime
import time

from datetime import datetime, timedelta
from timeit import default_timer as timer
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api(timeout=120)

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "16g",
    "spark.executor.memoryOverhead": "16g",
    "spark.reducer.maxReqsInFlight": "1",
    "spark.shuffle.io.retryWait": "60s",
    "spark.shuffle.io.maxRetries": "10",
}

# config = {
#     # "spark.sql.adaptive.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
#     # "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
#     "spark.executor.memory": "16g",
#     "spark.executor.memoryOverhead": "16g",
# }

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)
#api.start_spark(n_executors=400)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/05/02 16:16:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/05/02 16:16:13 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
24/05/02 16:17:19 WARN DFSClient: Slow waitForAckedSeqno took 65243ms (threshold=30000ms)
24/05/02 16:18:34 WARN DFSClient: Slow waitForAckedSeqno took 75133ms (threshold=30000ms)
24/05/02 16:18:41 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [5]:
api.spark.sparkContext.setLogLevel("ERROR")

In [6]:
year0 = "2023"
year1 = str(int(year0) + 1)

In [7]:
datelist = [
    year0 + "0101",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0301",
    year0 + "0501",
    year0 + "0701",
    year0 + "0901",
    year0 + "1101",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]

In [8]:
script = "retrieve_traffic_v4.ipynb"

In [9]:
i = 0
do_litetracks = True
do_flightplans = True
do_airspace_assignments = True

In [10]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230101', 'end_date': '20230301'}
start time:  2024-05-02 16:18:41.760393
starting:  {'start_date': '20230101', 'end_date': '20230301'}
   retrieving litetracks:  2024-05-02 16:18:41.893975


Multiple versions found: 3.1.51, 3.1.53, 3.1.56, 3.1.64


   retrieving planned_routes:  2024-05-02 16:19:12.158658


Multiple versions found: 3.1.49, 3.1.50, 3.1.51
Multiple versions found: 3.1.51, 3.1.52, 3.1.53, 3.1.65                         


   retrieving flightplans:  2024-05-02 16:19:21.166821


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 16:19:38.213762
   joining in df_plannedroutes:  2024-05-02 16:19:38.226665
   joining df_airspaceassignments:  2024-05-02 16:19:38.244785
   creating master dataframe:  2024-05-02 16:19:38.440811


      df_master size: 1772693 2024-05-02 16:23:33.482906
   started saving litetracks:  2024-05-02 16:23:34.142378


   completed saving litetracks:  2024-05-02 16:36:15.260213
   started saving flightplans:  2024-05-02 16:36:15.262932
24/05/02 16:36:15 ERROR TransportResponseHandler: Still have 1 requests outstanding when connection from /192.168.164.202:40054 is closed
24/05/02 16:36:15 ERROR TransportRequestHandler: Error sending result RpcResponse[requestId=6291491784710572459,body=NioManagedBuffer[buf=java.nio.HeapByteBuffer[pos=0 lim=47 cap=64]]] to /192.168.164.202:40054; closing connection
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/05/02 16:36:15 ERROR TransportClient: Failed to send RPC RPC 6653437017884906851 to /192.168.164.213:36076: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)


   completed saving flightplans:  2024-05-02 16:54:28.089567
   started saving airspace_assignments:  2024-05-02 16:54:28.094678


   completed saving airspace_assignments:  2024-05-02 16:58:50.214748
end time:  2024-05-02 16:58:50.215362


In [ ]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [12]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230301', 'end_date': '20230501'}
start time:  2024-05-02 17:00:42.724941
starting:  {'start_date': '20230301', 'end_date': '20230501'}
   retrieving litetracks:  2024-05-02 17:00:42.733610


Multiple versions found: 3.1.53, 3.1.54, 3.1.55, 3.1.56


   retrieving planned_routes:  2024-05-02 17:00:43.942020


Multiple versions found: 3.1.51, 3.1.53, 3.1.54
Multiple versions found: 3.1.52, 3.1.53, 3.1.54, 3.1.56                         


   retrieving flightplans:  2024-05-02 17:00:46.643539


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 17:00:58.117450
   joining in df_plannedroutes:  2024-05-02 17:00:58.127665
   joining df_airspaceassignments:  2024-05-02 17:00:58.141748
   creating master dataframe:  2024-05-02 17:00:58.324885


      df_master size: 2000286 2024-05-02 17:04:52.692275
   started saving litetracks:  2024-05-02 17:04:53.236087


   completed saving litetracks:  2024-05-02 17:19:13.909833
   started saving flightplans:  2024-05-02 17:19:13.916152


   completed saving flightplans:  2024-05-02 17:20:54.016646
   started saving airspace_assignments:  2024-05-02 17:20:54.018728


   completed saving airspace_assignments:  2024-05-02 17:22:22.782756
end time:  2024-05-02 17:22:22.783171


In [13]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230501', 'end_date': '20230701'}
start time:  2024-05-02 18:29:39.964430
starting:  {'start_date': '20230501', 'end_date': '20230701'}
   retrieving litetracks:  2024-05-02 18:29:39.973188


Multiple versions found: 3.1.55, 3.1.57, 3.1.58, 3.1.59


   retrieving planned_routes:  2024-05-02 18:29:54.426710


Multiple versions found: 3.1.54, 3.1.55, 3.1.58
Multiple versions found: 3.1.55, 3.1.56, 3.1.57, 3.1.58                         


   retrieving flightplans:  2024-05-02 18:30:00.360643


...

      df_master size: 2020920 2024-05-02 18:34:20.059968
   started saving litetracks:  2024-05-02 18:34:20.581768


   completed saving litetracks:  2024-05-02 18:47:26.572831
   started saving flightplans:  2024-05-02 18:47:26.574943


24/05/02 18:47:28 ERROR TransportClient: Failed to send RPC RPC 7510986661834329360 to /192.168.164.101:36760: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)


   completed saving flightplans:  2024-05-02 19:02:52.178302
   started saving airspace_assignments:  2024-05-02 19:02:52.184590


   completed saving airspace_assignments:  2024-05-02 19:08:38.150163
end time:  2024-05-02 19:08:38.150535


In [14]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230701', 'end_date': '20230901'}
start time:  2024-05-02 19:08:38.157964
starting:  {'start_date': '20230701', 'end_date': '20230901'}
   retrieving litetracks:  2024-05-02 19:08:38.163657


Multiple versions found: 3.1.59, 3.1.60, 3.1.61, 3.1.62


   retrieving planned_routes:  2024-05-02 19:08:41.681926


Multiple versions found: 3.1.58, 3.1.59, 3.1.60
Multiple versions found: 3.1.59, 3.1.60, 3.1.61


   retrieving flightplans:  2024-05-02 19:08:46.915326


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 19:09:16.818981
   joining in df_plannedroutes:  2024-05-02 19:09:16.826432
   joining df_airspaceassignments:  2024-05-02 19:09:16.838728
   creating master dataframe:  2024-05-02 19:09:16.982541


      df_master size: 2071563 2024-05-02 19:12:22.468596
   started saving litetracks:  2024-05-02 19:12:23.011897


   completed saving litetracks:  2024-05-02 19:24:01.005997
   started saving flightplans:  2024-05-02 19:24:01.011723


   completed saving flightplans:  2024-05-02 19:25:27.270199
   started saving airspace_assignments:  2024-05-02 19:25:27.272239


   completed saving airspace_assignments:  2024-05-02 19:27:52.790878
end time:  2024-05-02 19:27:52.791512


In [16]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230901', 'end_date': '20231101'}
start time:  2024-05-02 19:46:09.501348
starting:  {'start_date': '20230901', 'end_date': '20231101'}
   retrieving litetracks:  2024-05-02 19:46:09.510195


Multiple versions found: 3.1.61, 3.1.62, 3.1.63


   retrieving planned_routes:  2024-05-02 19:46:11.280564


Multiple versions found: 3.1.60, 3.1.62
Multiple versions found: 3.1.60, 3.1.61, 3.1.62, 3.1.63


   retrieving flightplans:  2024-05-02 19:46:13.709172


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 19:46:25.298443
   joining in df_plannedroutes:  2024-05-02 19:46:25.306144
   joining df_airspaceassignments:  2024-05-02 19:46:25.318309
   creating master dataframe:  2024-05-02 19:46:25.467246


      df_master size: 2028838 2024-05-02 19:50:19.278258
   started saving litetracks:  2024-05-02 19:50:19.772656


   completed saving litetracks:  2024-05-02 20:03:32.309991
   started saving flightplans:  2024-05-02 20:03:32.312144


   completed saving flightplans:  2024-05-02 20:04:55.093170
   started saving airspace_assignments:  2024-05-02 20:04:55.098278


   completed saving airspace_assignments:  2024-05-02 20:06:16.320586
end time:  2024-05-02 20:06:16.320938


In [17]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20231101', 'end_date': '20240101'}
start time:  2024-05-02 20:06:16.327734
starting:  {'start_date': '20231101', 'end_date': '20240101'}
   retrieving litetracks:  2024-05-02 20:06:16.333910


Multiple versions found: 3.1.62, 3.1.63, 3.1.64, 3.1.65


   retrieving planned_routes:  2024-05-02 20:06:18.831674


Multiple versions found: 3.1.62, 3.1.63, 3.1.64
Multiple versions found: 3.1.62, 3.1.63, 3.1.64, 3.1.65


   retrieving flightplans:  2024-05-02 20:06:22.265385


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 20:06:30.519130
   joining in df_plannedroutes:  2024-05-02 20:06:30.528110
   joining df_airspaceassignments:  2024-05-02 20:06:30.540347
   creating master dataframe:  2024-05-02 20:06:30.703035


      df_master size: 2015091 2024-05-02 20:09:38.823034
   started saving litetracks:  2024-05-02 20:09:39.313741


   completed saving litetracks:  2024-05-02 20:19:42.989830
   started saving flightplans:  2024-05-02 20:19:42.995483


   completed saving flightplans:  2024-05-02 20:21:09.805318
   started saving airspace_assignments:  2024-05-02 20:21:09.807186


   completed saving airspace_assignments:  2024-05-02 20:22:33.583620
end time:  2024-05-02 20:22:33.584176


In [ ]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [ ]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [ ]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [ ]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

In [ ]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())